In [ ]:
# Install required libraries
!pip install opencv-python pupil-apriltags pyusb matplotlib

In [ ]:
!conda install -y -c conda-forge libusb

Install https://zadig.akeo.ie/ and replace drive for launcher

In [ ]:
import usb.core
import usb.util

print("Scanning for USB devices...")
# Find all devices
devs = usb.core.find(find_all=True)

for cfg in devs:
    try:
        # Get hex strings for IDs
        vid = hex(cfg.idVendor)
        pid = hex(cfg.idProduct)
        print(f"Device: VID={vid}, PID={pid}")
        
        # Try to print the name (might fail if permissions are tight)
        # If this fails, just look for the VID/PID in the list
    except:
        pass

print("\n--- INSTRUCTIONS ---")
print("1. Look for a device that wasn't there before you plugged it in.")
print("2. 'Rocket Baby' often uses VID=0x0a81 and PID=0x0701 or similar.")
print("3. Replace the IDs in your main script with the ones you found.")

In [1]:
import cv2
import usb.core
import usb.util
from pupil_apriltags import Detector
import matplotlib.pyplot as plt

print("OpenCV Version:", cv2.__version__)
print("USB Library:", usb.__name__)

# Quick check if the launcher is plugged in and visible
# Vendor ID 0x2123 is standard for Dream Cheeky
dev = usb.core.find(idVendor=0xa81)
if dev:
    print("SUCCESS: Dream Cheeky Launcher detected!")
else:
    print("WARNING: Launcher not detected. (Check USB connection or drivers)")

OpenCV Version: 4.12.0
USB Library: usb
SUCCESS: Dream Cheeky Launcher detected!


In [ ]:
import usb.core
import usb.util
import time

VID = 0x0a81
PID = 0x0701

print(f"Connecting to {hex(VID)}:{hex(PID)}...")
dev = usb.core.find(idVendor=VID, idProduct=PID)

if not dev:
    print("Device not found! (Unplug/Replug)")
else:
    # 1. SETUP
    try:
        dev.set_configuration()
    except:
        pass

    # 2. DEFINING THE "ONE SHOT" COMMAND
    # We send the command ONCE. If the launcher moves for >0.1s, 
    # it means we don't need to spam it (which saves USB power).
    def move_oneshot(cmd_code, name):
        print(f">>> {name}...", end="")
        try:
            # 1. Send "Unlock" / Safety Clear (sending 0 to 0x0200)
            dev.ctrl_transfer(0x21, 0x09, 0x0200, 0, [0])
            time.sleep(0.05)
            
            # 2. Send Move Command
            dev.ctrl_transfer(0x21, 0x09, 0x0200, 0, [cmd_code])
            
            # 3. Wait to see if it sustains movement
            time.sleep(0.5) 
            
            # 4. Stop
            dev.ctrl_transfer(0x21, 0x09, 0x0200, 0, [32])
            print(" STOP.")
            time.sleep(0.5) # Cooldown
            
        except Exception as e:
            print(f" ERROR: {e}")

    try:
        # TEST 1: RIGHT (In case Left is stuck against the wall)
        move_oneshot(8, "RIGHT")

        # TEST 2: UP (gravity helps verify power)
        move_oneshot(2, "UP")

        # TEST 3: DOWN
        move_oneshot(1, "DOWN")
        
        # TEST 4: LEFT
        move_oneshot(4, "LEFT")

    except KeyboardInterrupt:
        dev.ctrl_transfer(0x21, 0x09, 0x0200, 0, [32])

In [3]:
import usb.core
import usb.util
import time
import sys

# Protocol Command Bytes
DOWN  = 0x01
UP    = 0x02
LEFT  = 0x04
RIGHT = 0x08
FIRE  = 0x10
STOP  = 0x20

# Global device variables
DEVICE = None
DEVICE_TYPE = None

def setup_usb():
    global DEVICE, DEVICE_TYPE
    
    # Try finding the newer "Thunder" model first
    DEVICE = usb.core.find(idVendor=0x2123, idProduct=0x1010)
    
    # If not found, look for your specific "Original" model
    if DEVICE is None:
        DEVICE = usb.core.find(idVendor=0x0a81, idProduct=0x0701)
        if DEVICE is None:
            raise ValueError('Missile device not found! Check connections or libusb drivers.')
        else:
            DEVICE_TYPE = "Original"
            print("Connected to: Original Dream Cheeky (VID: 0x0a81)")
    else:
        DEVICE_TYPE = "Thunder"
        print("Connected to: Dream Cheeky Thunder")

    # On Linux, detach the kernel driver if active
    if sys.platform == 'linux':
        try:
            if DEVICE.is_kernel_driver_active(0):
                DEVICE.detach_kernel_driver(0)
        except Exception as e:
            print(f"Driver detach warning: {e}")

    # Set configuration
    try:
        DEVICE.set_configuration()
        print("Device Configured Successfully.")
    except Exception as e:
        print(f"Error setting configuration: {e}")

# Run setup immediately
try:
    setup_usb()
except ValueError as e:
    print(e)

Connected to: Original Dream Cheeky (VID: 0x0a81)
Device Configured Successfully.


In [4]:
def send_cmd(cmd):
    if DEVICE is None:
        print("Device not connected.")
        return

    if "Thunder" == DEVICE_TYPE:
        # Newer Thunder Protocol
        DEVICE.ctrl_transfer(0x21, 0x09, 0, 0, [0x02, cmd, 0x00,0x00,0x00,0x00,0x00,0x00])
    
    elif "Original" == DEVICE_TYPE:
        # Your Protocol (VID 0xa81)
        # 0x21 = Request Type (Host to Device)
        # 0x09 = Request (Set Report)
        # 0x0200 = Value (Report Type/ID)
        DEVICE.ctrl_transfer(0x21, 0x09, 0x0200, 0, [cmd])

def send_move(cmd, duration_ms):
    send_cmd(cmd)
    time.sleep(duration_ms / 1000.0)
    send_cmd(STOP)

In [5]:
def move_up(duration=500):
    """Move Up for X milliseconds"""
    print(f"Moving UP for {duration}ms...")
    send_move(UP, duration)

def move_down(duration=500):
    """Move Down for X milliseconds"""
    print(f"Moving DOWN for {duration}ms...")
    send_move(DOWN, duration)

def move_left(duration=500):
    print(f"Moving LEFT for {duration}ms...")
    send_move(LEFT, duration)

def move_right(duration=500):
    print(f"Moving RIGHT for {duration}ms...")
    send_move(RIGHT, duration)

def fire_missiles(count=1):
    if count < 1 or count > 4:
        count = 1
    print(f"Firing {count} missile(s)...")
    # Stabilize
    time.sleep(0.5)
    for i in range(count):
        send_cmd(FIRE)
        # Wait for reload cycle (approx 4.5s)
        time.sleep(4.5)

In [6]:
# TEST 1: Verify Left/Right works (you said this clicked)
move_right(1000)
time.sleep(1)
move_left(1000)

# TEST 2: The Silent Motor
# Listen closely to the unit while running this.
# If completely silent -> Disconnected wire or dead motor.
# If humming/whining -> Stuck gears.
print("\nTesting Elevation Motor...")
move_up(2000)
time.sleep(1)
move_down(2000)

Moving RIGHT for 1000ms...
Moving LEFT for 1000ms...

Testing Elevation Motor...
Moving UP for 2000ms...
Moving DOWN for 2000ms...
